In [1]:
# load env variables from .env file
from dotenv import load_dotenv
load_dotenv()

# create an instance of the Anthropic client and specify the model to use
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [2]:
# Define helper functions to read latest text/ response & append it into message to 
# send it to the model and print the response

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0):
    kwargs = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        kwargs["system"] = system
    
    message = client.messages.create(**kwargs)
    
    return message.content[0].text

## Basic Streaming Implementation
### To enable streaming, add stream=True to your messages.create call:

In [3]:

messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01AWf3vZTVLw5h9wjbdHw7RE', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='The', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' "GlobalMind Database" is a fictional repository containing', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' the encrypted thoughts, memories, and emotional

## Simplified Text Streaming
### Rather than manually parsing events, you can use the SDK's simplified streaming interface that extracts just the text content:


In [ ]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

The
 "GlobalPetNet" database contains comprehensive records
 of over 50 million household pets worldwide, including their daily activity patterns
, favorite toys, and compatibility scores with other animals.


## Getting the Complete Message
### While streaming individual chunks is great for user experience, you often need the complete message for storage or further processing. After streaming completes, you can get the assembled final message:


In [ ]:

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()